# Anexo — Evaluación de la selección de HVG (notebook 02)

En el notebook 02 se vió que la selección de genes altamente variables se calculó sobre adata.X ya normalizado, en vez de sobre los conteos crudos. Se calcula cuánto cambia la lista de genes resultante, y si eso pudo afectar a las conclusiones de los notebooks siguientes.

La limitación se describe y discute en el apartado de **Limitaciones** de la memoria.


## Carga del objeto normalizado

In [ ]:
# Montar Google Drive y prepara el gestor de entornos Conda,
# que aísla las dependencias del proyecto del entorno base de Colab.
from google.colab import drive
drive.mount('/content/drive')

!pip install -q condacolab
import condacolab
condacolab.install()

Mounted at /content/drive
⏬ Downloading https://github.com/conda-forge/miniforge/releases/download/25.11.0-1/Miniforge3-25.11.0-1-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:14
🔁 Restarting kernel...


In [ ]:
# Instalación del entorno Conda del proyecto (environment.yml), con las
# versiones exactas de todas las dependencias fijadas para reproducibilidad.

!conda env update -n base -f /content/drive/MyDrive/TFM_IBD_GSE214695/environment.yml -q

Retrieving notices: ...working... done
Channels:
 - conda-forge
 - bioconda
 - defaults
Platform: linux-64
Solving environment: ...working... done
Preparing transaction: ...working... done
Verifying transaction: ...working... done
Executing transaction: ...working... done
Installing pip dependencies: ...working... done


In [ ]:
import scanpy as sc
import numpy as np
from scipy.stats import spearmanr

PATH = "/content/drive/MyDrive/IBD_TFM/data/interim/02_normalized/IBD_normalized.h5ad"
adata = sc.read_h5ad(PATH)
print(adata)

/usr/local/lib/python3.12/site-packages/scanpy/_utils/__init__.py:27: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/usr/local/lib/python3.12/site-packages/scanpy/__init__.py:36: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/usr/local/lib/python3.12/site-packages/scanpy/readwrite.py:15: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):


AnnData object with n_obs × n_vars = 43388 × 33538
    obs: 'sample_id', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'pct_counts_ribo', 'log10_ratio', 'doublet_score', 'predicted_doublet'
    var: 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'highly_variable_nbatches'
    uns: 'hvg', 'log1p', 'pca'
    obsm: 'X_pca'


## 1. Confirmar que `adata.raw` contiene conteos crudos reales


In [ ]:
raw_sample = adata.raw.X[:5, :10].toarray() if hasattr(adata.raw.X, "toarray") else adata.raw.X[:5, :10]
x_sample   = adata.X[:5, :10].toarray() if hasattr(adata.X, "toarray") else adata.X[:5, :10]

print("adata.raw.X:")
print(np.round(raw_sample, 3))

print("\nadata.X:")
print(np.round(x_sample, 3))

print(f"\nmax raw.X: {adata.raw.X.max()}  |  max X: {adata.X.max():.3f}")

adata.raw.X:
[[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]

adata.X:
[[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]

max raw.X: 35077.0  |  max X: 9.130


## 2. Recalcular la selección de HVG de las dos formas

- **`a_correcto`**: `flavor='seurat_v3'` sobre los conteos crudos.
- **`a_v3style`**: `flavor='seurat_v3'` sobre `adata.X` (ya normalizado), reproduciendo el notebook 02 entregado.



In [ ]:
N_HVG = 3000
BATCH_KEY = "sample_id"

a_correcto = adata.copy()
a_correcto.layers['counts'] = a_correcto.raw.to_adata().X.copy()
sc.pp.highly_variable_genes(
    a_correcto, layer='counts', n_top_genes=N_HVG,
    flavor='seurat_v3', batch_key=BATCH_KEY, subset=False
)

a_v3style = adata.copy()
sc.pp.highly_variable_genes(
    a_v3style, n_top_genes=N_HVG,
    flavor='seurat_v3', batch_key=BATCH_KEY, subset=False
)

hvg_correcto = set(a_correcto.var_names[a_correcto.var['highly_variable']])
hvg_v3style  = set(a_v3style.var_names[a_v3style.var['highly_variable']])

/usr/local/lib/python3.12/site-packages/scanpy/preprocessing/_highly_variable_genes.py:75: UserWarning: `flavor='seurat_v3'` expects raw count data, but non-integers were found.
  warnings.warn(


## 3. Solapamiento entre ambas listas de HVG

In [ ]:
overlap = hvg_correcto & hvg_v3style
print(f"Solapamiento top-{N_HVG}: {len(overlap)}/{N_HVG} ({100*len(overlap)/N_HVG:.2f}%)")
print(f"Solo en versión correcta: {len(hvg_correcto - hvg_v3style)}")
print(f"Solo en versión v3-style: {len(hvg_v3style - hvg_correcto)}")

Solapamiento top-3000: 1100/3000 (36.67%)
Solo en versión correcta: 1900
Solo en versión v3-style: 1900


## 4. Correlación del ranking completo

In [ ]:
rank_correcto = a_correcto.var['highly_variable_rank']
rank_v3style  = a_v3style.var['highly_variable_rank']
mask = rank_correcto.notna() & rank_v3style.notna()

rho, p = spearmanr(rank_correcto[mask], rank_v3style[mask])
print(f"Correlación de Spearman entre rankings completos: {rho:.4f} (n={mask.sum()}, p={p:.2e})")

Correlación de Spearman entre rankings completos: 0.3744 (n=8144, p=1.98e-269)


## 5. Concordancia con los marcadores canónicos usados en la anotación



In [ ]:
canonicos = ['EPCAM','KRT20','MUC2','TFF3','CD3D','CD3E','CD8A','CD4','CD79A','MS4A1',
             'IGHA1','MZB1','C1QA','CD68','LYZ','S100A8','COL1A1','DCN','VWF','PECAM1',
             'TPSAB1','CPA3','POU2F3','TRPM5']

print(f"{'Gen':<10} {'Crudo (correcto)':<18} {'X normalizado (con error)':<26}")
faltan_en_correcto, faltan_en_v3 = [], []
for g in canonicos:
    en_c = g in hvg_correcto
    en_v = g in hvg_v3style
    print(f"{g:<10} {str(en_c):<18} {str(en_v):<26}")
    if not en_c: faltan_en_correcto.append(g)
    if not en_v: faltan_en_v3.append(g)

print(f"\nFaltan en versión correcta (cruda): {faltan_en_correcto}")
print(f"Faltan en versión con el error: {faltan_en_v3}")

# Resultado obtenido: 20/24 coinciden en ambas versiones.
# CD4 falta en ambas (no es efecto del error).
# CD79A, CD68, PECAM1 solo faltan en la versión con el error, pero B cell (MS4A1),
# Macrófago (C1QA, LYZ) y Endotelio (VWF) conservan al menos otro marcador
# canónico presente en ambas versiones.

Gen        Crudo (correcto)   X normalizado (con error) 
EPCAM      True               True                      
KRT20      True               True                      
MUC2       True               True                      
TFF3       True               True                      
CD3D       True               True                      
CD3E       True               True                      
CD8A       True               True                      
CD4        False              False                     
CD79A      True               False                     
MS4A1      True               True                      
IGHA1      True               True                      
MZB1       True               True                      
C1QA       True               True                      
CD68       True               False                     
LYZ        True               True                      
S100A8     True               True                      
COL1A1     True               T

## 6. Conclusión

- Solapamiento de HVG: **36.67%** (1.100/3.000).
- Correlación de Spearman del ranking completo: **0.37**
- Marcadores canónicos: **20/24 coinciden**; los 3 que se pierden tienen un marcador sustituto del mismo linaje en ambas versiones.
